# Faithfulness summary tables

Set `DEBUG = True` in the setup cell to read from `Temp/Results`.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Daprosero/STF-KernelSHAP.git"
REPO_NAME = "STF-KernelSHAP"
PACKAGE_PATH = Path("src") / "stf_kernelshap"


def resolve_project_root():
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / PACKAGE_PATH).exists():
            return candidate

    working_root = Path("/kaggle/working") if Path("/kaggle/working").exists() else cwd
    clone_dir = working_root / REPO_NAME
    if not clone_dir.exists():
        subprocess.run(["git", "clone", REPO_URL, str(clone_dir)], check=True)
    return clone_dir.resolve()


def install_project_requirements(project_root):
    requirements_path = project_root / "requirements.txt"
    if not requirements_path.exists():
        raise FileNotFoundError(f"No existe requirements.txt en {project_root}")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements_path)],
        check=True,
    )


PROJECT_ROOT = resolve_project_root()
os.chdir(PROJECT_ROOT)

RUNNING_IN_COLAB = "COLAB_RELEASE_TAG" in os.environ
RUNNING_IN_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ or Path("/kaggle").exists()
INSTALL_REQUIREMENTS = RUNNING_IN_COLAB or RUNNING_IN_KAGGLE

if INSTALL_REQUIREMENTS:
    install_project_requirements(PROJECT_ROOT)

SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("INSTALL_REQUIREMENTS:", INSTALL_REQUIREMENTS)


In [ ]:
from stf_kernelshap.notebook_setup import setup_notebook_environment

DEBUG = True
paths = setup_notebook_environment(debug=DEBUG)

DATA_DIR = paths.data_dir
MODELS_DIR = paths.models_dir
OUTPUT_MODELS_DIR = paths.output_models_dir
RESULTS_DIR = paths.results_dir
FIGURES_DIR = paths.figures_dir


In [ ]:
import pandas as pd

from stf_kernelshap.reporting.faithfulness_summary import (
    aggregate_auc_by_window_model_method,
    compute_average_rank_by_window_metric,
    compute_faithfulness_auc_summary,
    get_best_only_by_window,
    print_rankings_for_window,
)


In [ ]:
plot_ready_csv = RESULTS_DIR / "faithfulness_selected_relevance" / "faithfulness_selected_relevance_score_plot_ready.csv"
df_plot_ready = pd.read_csv(plot_ready_csv)

df_auc = compute_faithfulness_auc_summary(df_plot_ready)
df_auc_summary = aggregate_auc_by_window_model_method(df_auc)
df_average_rank = compute_average_rank_by_window_metric(df_auc_summary)


In [ ]:
rankings = {
    window_name: print_rankings_for_window(df_auc_summary, window_name)
    for window_name in ("2.5-5", "0-7", "TDAH")
}

best_by_window = {
    window_name: get_best_only_by_window(df_auc_summary, window_name)
    for window_name in ("2.5-5", "0-7", "TDAH")
}
